In [ ]:
import pandas as pd

from tms_risk.utils import get_subjects, get_tms_conditions, get_all_behavior
from tqdm.contrib.itertools import product
import seaborn as sns
import pingouin
import numpy as np
import matplotlib.pyplot as plt
import os.path as op
import pingouin as pg

bids_folder  = '/data/ds-tmsrisk'

stimulation_palette = sns.color_palette()[2:4]
stimulation_order = ['Vertex', 'IPS']

# Only TMS cluster

In [ ]:
subjects = get_subjects(all_tms_conditions=True)

In [ ]:
pars = []
keys = []

for sub, session, roi in product(subjects, [2,3], ['NPCr2cm-cluster']):

    try:
        pars.append(sub.get_prf_parameters(model_label=1, session=session, roi=roi))  # was: sub.get_prf_parameters_volume(session, smoothed=True, retroicor=False, denoise=T...)
        keys.append((sub.subject, session, roi))
    except Exception as e:
        print(e)

pars = pd.concat(pars, keys=keys, names=['subject', 'session', 'roi'])
tms_conditions = get_tms_conditions()
for key in tms_conditions:
    tms_conditions[key][1] = 'baseline'

pars['stimulation_condition'] = pars.reset_index().apply(lambda d: tms_conditions[d['subject']][d['session']],  axis=1).values
pars = pars.set_index('stimulation_condition', append=True)
pars.index = pars.index.set_names('voxel', level=-2)


In [ ]:
pars_ = pars.copy()

* Find distribution of mus
* Find amplitudes as function of mus
* Find R2 as a function of mus

# Use only pars with CVr > 0.0 _within session_ and then see effect of brain sitmulation

In [ ]:
cvr2 = pars.droplevel('session')['cvr2'].unstack('stimulation_condition')
mask = (cvr2 > 0.0).any(axis=1)

In [ ]:
# thr_pars = pars[pars.cvr2 > -0.01]
thr_pars = pars.droplevel('session').loc[mask]

tmp = thr_pars.groupby(['subject', 'stimulation_condition']).mean().stack().to_frame('value')

In [ ]:
from IPython.display import display

In [ ]:
# thr_pars['mu_natural'] = np.exp(thr_pars['mu'])
thr_pars['mu_natural'] = thr_pars['mu']

thr_pars['out_of_sample_cvr2_postive'] = thr_pars['cvr2'] > 0.0
thr_pars.groupby(['subject', 'stimulation_condition']).mean().groupby('stimulation_condition').median()

In [ ]:
tmp = thr_pars.groupby(['subject', 'stimulation_condition']).mean().stack().to_frame('value')

for par, tmp2 in tmp.groupby('parameter'):
    #  tmp.drop('baseline', level='stimulation_condition').xs('amplitude', 0, 'parameter').groupby(['subject', 'stimulation_condition']).mean().reset_index()

    ax = sns.catplot(x='stimulation_condition', y='value', data=tmp2.reset_index(), kind='bar')
    plt.gcf().suptitle(par)
    # print(pingouin.rm_anova(tmp2.reset_index(), 'value', 'stimulation_condition', 'subject'))
    print(par)
    display(pingouin.pairwise_tests(data=tmp2.reset_index(), dv='value', within='stimulation_condition', subject='subject', alternative='two-sided'))


In [ ]:
tmp = (pars['cvr2'] > 0.0).groupby(['subject', 'stimulation_condition']).mean().reset_index()

sns.catplot(x='stimulation_condition', y='cvr2', data=tmp.reset_index(), kind='bar')

display(tmp.groupby('stimulation_condition')['cvr2'].mean())


tmp = tmp.set_index(['subject', 'stimulation_condition']).unstack('stimulation_condition')['cvr2']

display(pg.ttest(tmp['ips'], tmp['vertex'], paired=True, alternative='less'))

In [ ]:
thr_pars.to_csv(op.join(bids_folder, 'derivatives', 'encoding_models', 'prf_parameters_thr.tsv'), sep='\t')

In [ ]:
sns.kdeplot(np.clip(thr_pars['mu_natural'], 0, 50))

In [ ]:
thr_pars['log_mu'] = thr_pars['mu']
thr_pars['mu_natural'] = np.exp(thr_pars['mu'])
# thr_pars_ = thr_pars[thr_pars['mu'] <]

# thr_pars = thr_parsd
thr_pars['mu_bin'] = pd.cut(thr_pars['mu_natural'], bins=np.arange(0, 50, 5.))

# Convert to midpoint
thr_pars['mu_bin'] = thr_pars['mu_bin'].apply(lambda x: x.mid)

tmp = thr_pars.groupby(['subject', 'stimulation_condition', 'mu_bin']).mean().reset_index()
tmp['stimulation_condition'] = tmp['stimulation_condition'].map({'vertex':'Vertex', 'ips':'IPS'})

g = sns.relplot(x='mu_bin', y='amplitude', hue='stimulation_condition', data=tmp.reset_index(), kind='line', errorbar='se', palette=stimulation_palette, hue_order=stimulation_order, height=3., aspect=1.25, legend=False)
g.set(ylim=(0, None))

g = sns.relplot(x='mu_bin', y='r2', hue='stimulation_condition', data=tmp.reset_index(), kind='line', errorbar='se', palette=stimulation_palette, hue_order=stimulation_order, height=3., aspect=1.25, legend=False)
g.set(ylim=(0, None))

In [ ]:
from tms_risk.utils.data import get_all_behavior

df = get_all_behavior()
sns.histplot(thr_pars['mu'], bins=100, kde=False, stat='density')
sns.histplot(df['n1'], bins=100, kde=False, stat='density')
sns.despine()

In [ ]:
from matplotlib.gridspec import GridSpec

# Set all font sizes to 14
sns.set(font_scale=1.6, style='white', font='Helvetica')

tmp = thr_pars.copy()
tmp = tmp[tmp['mu_natural'] < 50]
tmp['mu_natural'] = np.exp(tmp['mu'])
tmp['mu_bin'] = pd.cut(tmp['mu_natural'], bins=np.arange(-5, 35, 5.))
# tmp['mu_bin'] = pd.qcut(tmp['mu_natural'], q=10)
tmp['mu_bin'] = tmp['mu_bin'].apply(lambda x: x.mid)

tmp['cvr2_positive'] = tmp['cvr2'] > 0.0

tmp = tmp.groupby(['subject', 'stimulation_condition', 'mu_bin']).median().reset_index()

fig = plt.figure(figsize=(8, 6))
gs = GridSpec(2, 1, height_ratios=[5, 3])  # 2:1 height ratio

ax_top = fig.add_subplot(gs[0, 0])

# sns.lineplot(x='mu_bin', y='r2', hue='stimulation_condition', data=tmp.reset_index(), ax=ax_top, palette=stimulation_palette, hue_order=['vertex', 'ips'])
sns.lineplot(x='mu_bin', y='amplitude', hue='stimulation_condition', data=thr_pars.reset_index(), ax=ax_top, palette=stimulation_palette, hue_order=['vertex', 'ips'])
sns.despine()
# Put the legend top middle
plt.legend(loc='upper left', bbox_to_anchor=(0.25, 1.), ncol=1)

# Capitalize two hue labels
leg = ax_top.get_legend()
for t in leg.texts:
    t.set_text(t.get_text().capitalize())

# ax_top.set_yticks([0.0, 0.05, 0.1,])
ax_top.set_ylabel('Amplitude')
ax_top.set_xlabel(None)


ax_bottom = fig.add_subplot(gs[1, 0], sharex=ax_top)

sns.histplot(thr_pars, x='mu_natural', element='step', ax=ax_bottom, color='gray', bins=np.arange(1, 100), stat='density', label='Preferred numerosities')
sns.kdeplot(df['n1'], color='k', shade=False, alpha=1., lw=2,ls='--', label='Stimulus distribution')
sns.despine()
ax_bottom.set_xlabel('Preferred numerosity')

ax_bottom.legend()
ax_bottom.set_xlim(0, 30)

plt.savefig(op.join(bids_folder, 'derivatives', 'figures', 'amplitude_vs_preferred_numerosity.pdf'), bbox_inches='tight')

In [ ]:
ax =g.axes[0, 0]
ax.plot([0,1], [0,1])

In [ ]:
tmp = (pars['cvr2'] > 0.0).groupby(['subject', 'stimulation_condition', 'mu']).mean().reset_index()
tmp

In [ ]:
thr_pars['log_mu'] = thr_pars['mu']
thr_pars['mu_natural'] = np.exp(thr_pars['mu'])
# thr_pars_ = thr_pars[thr_pars['mu'] <]
thr_pars = thr_pars[thr_pars['cvr2'] > 0.0]
thr_pars['mu_bin'] = pd.qcut(thr_pars['mu_natural'], q=5)

# Convert to midpoint
thr_pars['mu_bin'] = thr_pars['mu_bin'].apply(lambda x: x.mid)

tmp = thr_pars.groupby(['subject', 'stimulation_condition', 'mu_bin']).mean()

# Fit a misxed effefts model
from statsmodels.formula.api import mixedlm

model = mixedlm('amplitude ~ mu * stimulation_condition', thr_pars.reset_index(), groups=thr_pars.reset_index()['subject'])

result = model.fit()
result.summary()


# All ROIs

In [ ]:
pars = []
keys = []

for sub, session, roi in product(subjects, [2,3], ['NPC1l', 'NPC1r', 'NPC2l', 'NPC2r', 'NPC3l', 'NPC3r', 'NTOl', 'NTOr', 'NF1l', 'NF1r', 'NF2l', 'NF2r', 'NPCr1cm-cluster', 'NPCr2cm-cluster']):

    try:
        pars.append(sub.get_prf_parameters(model_label=1, session=session, roi=roi))  # was: sub.get_prf_parameters_volume(session, smoothed=True, retroicor=False, denoise=T...)
        keys.append((sub.subject, session, roi))
    except Exception as e:
        print(e)

pars = pd.concat(pars, keys=keys, names=['subject', 'session', 'roi'])
tms_conditions = get_tms_conditions()
for key in tms_conditions:
    tms_conditions[key][1] = 'baseline'

pars['stimulation_condition'] = pars.reset_index().apply(lambda d: tms_conditions[d['subject']][d['session']],  axis=1).values
pars = pars.set_index('stimulation_condition', append=True)
pars.index = pars.index.set_names('voxel', level=-2)


In [ ]:
cvr2 = pars.droplevel('session')['cvr2'].unstack('stimulation_condition')
mask = (cvr2 d> 0.0).any(axis=1)

In [ ]:
thr_pars = pars.droplevel('session').loc[mask]

tmp = thr_pars.groupby(['subject', 'stimulation_condition', 'roi']).mean().stack().to_frame('value')

In [ ]:
for par, tmp2 in tmp.groupby('parameter'):
    #  tmp.drop('baseline', level='stimulation_condition').xs('amplitude', 0, 'parameter').groupby(['subject', 'stimulation_condition']).mean().reset_index()

    ax = sns.catplot(x='roi', hue='stimulation_condition', y='value', data=tmp2.reset_index(), kind='bar', aspect=5.)
    plt.gcf().suptitle(par)
    # print(pingouin.rm_anova(tmp2.reset_index(), 'value', 'stimulation_condition', 'subject'))
    print(par)
    display(tmp2.groupby(['roi']).apply(lambda d: pingouin.pairwise_tests(d.reset_index(), dv='value', within='stimulation_condition', subject='subject')))

In [ ]:
summary = thr_pars.groupby(['roi', 'subject', 'stimulation_condition']).mean().groupby(['roi', 'stimulation_condition']).agg(['mean', 'std']).unstack(-1)[['mu', 'sd', 'amplitude']].astype(np.float64).round(2)

In [ ]:
summary.stack([0, 2])

In [ ]:
pars.groupby(['subject', 'stimulation_con'])

In [ ]:
summary.stack([0, 2]).apply(lambda row: f'{row["mean"]} ({row["std"]})', axis=1).unstack([-2, -1])